# TP 1 et 2 : Accès aux données avec index


date de modification : 16/01/2025

BINOME:

NOM 1:PARAT

Prénom 1:Nicolas

NOM 2:GOLDBERG

Prénom 2:Denis


Rapport à rendre : **Rédiger individuellement des explicatons détaillées** pour les solutions que vous proposez. Format du rapport : PDF de 1 à 3 pages




Objectifs:
Savoir organiser le stockage de données.
Savoir stocker les nuplets d'une relation dans un ensemble de pages.
Savoir lire et modifier des nuplets lorsqu'ils sont stockés dans des pages.

Comprendre les méthodes d'accès suivantes :

*   Lecture séquentielle d'une fichier : "table access full"
*   Lecture d'un tuple dont on connait le rowid : "table access by index rowid"
*   Opération de sélection par lecture séquentielle et filtrage

Comprendre les méthodes d'indexation :

*   Créer un index
*   Opération de Sélection par index et lecture par rowid

Mise à jour de données
*   Sélectionner un tuple et modifier un de ses attributs
*   Modifier l'index en conséquence lorsque l'attribut modifié est indexé

Persistence
*   Stocker un index (dans plusieurs pages) pour le reconstruire plus rapidement
*   Adapter en conséquence les opérations de modification de l'index


# Préparation du TP

In [1]:
!pip install -q sortedcontainers

In [2]:
import os
import glob
import shutil as sh
import numpy as np
import pandas as pd
import random

# from sortedcontainers import SortedDict
import sortedcontainers

from string import ascii_lowercase
import time

# le nom de la table
TABLE = "T"
print("le nom de la table est", TABLE)


# le nom du fichier qui contient les données de la table
def nom_fichier(table):
    return table + ".csv"


# Iterator
try:
    from collections.abc import Iterator  # For Python 3.9+
except ImportError:
    from typing import Iterator  # For Python 3.8 and below


le nom de la table est T


## Générer les données du TP

Création du fichier contenant un exemple de données.
Ce sont des données au format csv. On suppose que chaque ligne correspond à un tuple d'une table **T** ayant *n* attributs :

$$ T (a_0, a_1, a_2, ..., a_{n-1})$$

Le premier attribut $a_0$ est unique.

Les attributs $a_1$ à $a_{n-2}$ ne sont pas uniques : il y a en moyenne $2^k$ tuples par valeurs de $a_k$ soit 2 tuples par valeur de $a_1$ et 4 tuples par valeurs de $a_2$ , *etc.*

Les attributs (sauf le dernier) sont des nombres entiers multiples de 10 afin d'avoir des domaines contenant des valeurs *non* consécutives.
Le dernier attribut $a_{n-1}$ est une chaine de caractères.


In [3]:
from re import NOFLAG
# dure environ 10s pour 2M lignes
# dure environ 20s pour 5M lignes

# les attributs sont des multiples de 10
MULTIPLE = 10

NB_LIGNES = 2 * 1000 * 1000
# NB_LIGNES = 5 * 1000 * 1000

nb_attributs = 7
longueur_dernier_attribut = 100

# Ajouté par moi
nb_valeurs_distinctes = NB_LIGNES


def genere_fichier(nb_lignes, nb_attributs, longueur_dernier_attribut, table):
  # Ajouté par moi
  global nb_valeurs_distinctes

  # attribut_chaine_caracteres = "".join(choice(ascii_lowercase) for i in range(longueur_dernier_attribut))
  attribut_chaine_caracteres = ''.join(['-'] * longueur_dernier_attribut)
  # print("le dernier attribut de chaque tuple est la chaine de caracètes :", attribut_chaine_caracteres)

  # reproductibilité des données générées
  rng = np.random.default_rng(seed=1)

  data={}



  # le premier attribut est unique et multiple de 10.
  nb_valeurs_distinctes = nb_lignes

  data['a0'] = MULTIPLE * rng.permutation(np.arange(nb_valeurs_distinctes))

  # les attributs suivants ont des domaines plus petits :
  for i in range(1, nb_attributs):
    # on divise le domaine par 2 à chaque itération
    nb_valeurs_distinctes = max(2, nb_valeurs_distinctes // 2)
    data[f'a{i}'] = MULTIPLE * rng.integers(0, nb_valeurs_distinctes, nb_lignes)

  # on concatène "verticalement" les attributs dans un dataframe pour former des tuples sur lesquels on peut itérer.
  df = pd.DataFrame(data)

  # on stocke les données dans un fichier
  fichier = nom_fichier(table)
  print("écriture des données dans le fichier", fichier)

  #
  iter_lines = ( f"{str(e)[1:-1]}, {attribut_chaine_caracteres}" for e in df.itertuples(index=False, name=None) )
  with open(fichier, "w") as f:
    # écriture groupée de tous les tuples
    f.write('\n'.join(iter_lines))
    f.write('\n')



t1 = time.time()
genere_fichier(NB_LIGNES, nb_attributs, longueur_dernier_attribut, TABLE)
print(f"durée pour générer {NB_LIGNES} lignes :", round(time.time() - t1, 1), "s")

écriture des données dans le fichier T.csv
durée pour générer 2000000 lignes : 13.3 s


In [4]:
%%bash -s "{TABLE}.csv"
echo "debut de $1 :"
head -n 2 $1
echo
echo "fin de $1 : "
tail -n 2 $1
echo
echo "nombre de lignes:"
wc -l $1


debut de T.csv :
19512870, 5751570, 1235270, 1344040, 1139220, 359300, 261790, ----------------------------------------------------------------------------------------------------
3462320, 3047440, 4558650, 2392580, 332870, 445910, 35390, ----------------------------------------------------------------------------------------------------

fin de T.csv : 
4418430, 9611540, 268500, 660850, 881140, 311230, 273070, ----------------------------------------------------------------------------------------------------
1896950, 9749280, 4493840, 1757100, 1083240, 323240, 177530, ----------------------------------------------------------------------------------------------------

nombre de lignes:
2000000 T.csv


On affiche le début et la fin du fichier et son nombre de lignes ( = card(T))

## Lecture séquentielle

On définit un *iterateur* pour lire séquentiellement chaque ligne de la table stockée entièrement dans un seul fichier.
Le mot python *yield* permet de définir un itérateur qui est retourné par la fonction.

Cet itérateur est invoqué pour lire la table et appliquer un filtre.



In [5]:
def lecture_sequentielle(table) -> Iterator:
  fichier = nom_fichier(table)
  with open(fichier, "r") as f:
    for line in f:
      yield line.strip().split(',')

## Requête de sélection

In [6]:
def filtrer_table_par_pages(input_data: Iterator, attribut: int, valeur_recherchee: int) -> Iterator:
  for tuple_courant in input_data:
    if attribut < len(tuple_courant):
      valeur = int(tuple_courant[attribut])
      if valeur == valeur_recherchee :
        yield tuple_courant



search = MULTIPLE * np.random.randint(NB_LIGNES)
print("valeur recherchée pour a0 :", search)
t1 = time.time()
input_data = lecture_sequentielle(TABLE)
for t in filtrer_table_par_pages(input_data, 0, search):
  print(t)
print("durée :", round(time.time() - t1, 2), "s")


search = MULTIPLE * np.random.randint(NB_LIGNES/2)
print("valeur recherchée pour a1 :", search)
t1 = time.time()
input_data = lecture_sequentielle(TABLE)
for t in filtrer_table_par_pages(input_data, 1, search):
  print(t)
print("durée :", round(time.time() - t1, 2), "s")


valeur recherchée pour a0 : 9673440
['9673440', ' 8487050', ' 3255480', ' 845570', ' 861040', ' 488430', ' 238440', ' ----------------------------------------------------------------------------------------------------']
durée : 3.21 s
valeur recherchée pour a1 : 6095200
['13489460', ' 6095200', ' 1508720', ' 310850', ' 368900', ' 197290', ' 124720', ' ----------------------------------------------------------------------------------------------------']
durée : 3.17 s


## Découper une table en pages

On organise les données en pages.
Pour faciliter le TP, chaque page est représentée par un "petit" fichier mais en réalité une page est un bloc d'un fichier.

Dans la suite du TP, on accédera toujours aux pages.
Le fichier créé initialement, contenant tous les tuples, ne sera plus utilisé.

In [7]:
NB_TUPLES_PAR_PAGE = 1000


def page_dir_name(table):
  return table + "_pages"

def decoupe_table_en_pages(table, nb_tuple_par_page):
  page_dir = page_dir_name(table)
  print(f"une page contient {nb_tuple_par_page} tuples")

  # vider le dossier qui contiendra les pages
  if(os.path.exists(page_dir)):
    sh.rmtree(page_dir)
  os.makedirs(page_dir, exist_ok=True)

  # lire le fichier contenant tous les tuples
  p=0
  lines = []
  fichier = nom_fichier(table)
  with open(fichier, "r") as f:
    for line in f:
      lines.append(line)
      if len(lines) == nb_tuple_par_page:
        # créer une page
        p += 1
        with open(os.path.join(page_dir, f"page{p}"), "w") as fp:
          fp.write(''.join(lines))
        lines = []

  # créer une dernière page, si nécessaire
  if len(lines) > 0:
    print(f"la dernière page contient {len(lines)} tuples")
    p +=1
    with open(os.path.join(page_dir, f"page{p}"), "w") as fp:
        fp.write(''.join(lines))

  print("nb pages créées :", p)


print("les pages sont stockées dans le dossier", page_dir_name(TABLE) )

decoupe_table_en_pages(TABLE, NB_TUPLES_PAR_PAGE)

les pages sont stockées dans le dossier T_pages
une page contient 1000 tuples
nb pages créées : 2000


Afficher (pour quelques pages) le nombre de tuples contenus dans une page

une solution en bash :

In [53]:
%%bash -s "$TABLE"
wc -l $1_pages/* | head -n 3

     1000 T_pages/page1
     1000 T_pages/page10
     1000 T_pages/page100


une autre solution en python :

In [8]:
page_dir = page_dir_name(TABLE)
l = os.listdir(page_dir)
random.seed(1)
for i in range(3):
 une_page = random.choice(l)
 with open(page_dir + f"/{une_page}", 'r') as fp:
   lines = len(fp.readlines())
   print(f"la page {une_page} contient {lines} lignes")

la page page1138 contient 1000 lignes
la page page1511 contient 1000 lignes
la page page923 contient 1000 lignes


## Lecture séquentielle des pages

In [9]:
def lecture_sequentielle_par_page(table) -> Iterator:
  page_dir = page_dir_name(table)

  # nombre de pages
  nb_pages = len(glob.glob(os.path.join(page_dir, "page*")))

  # itérer sur les pages de la table
  # on suppose que les page sont numérotés de 1 à nb_pages
  for page_id in range(1, nb_pages+1) :
    with open(os.path.join(page_dir, f"page{page_id}"), "r") as f:
      for position, line in enumerate(f):
        tuple_courant = line.strip().split(',')
        yield tuple_courant

## Requête de sélection (page par page)

In [ ]:
# A COMPLETER

search = MULTIPLE * np.random.randint(NB_LIGNES)
print("valeur recherchée pour a0 :", search)

# lecture séquentielle par page
input_data = ................................(TABLE)
t1 = time.time()
for t in filtrer_table_par_pages(input_data, 0, search):
  print(t)
print("durée :", round(time.time() - t1, 2), "s")


search = MULTIPLE * np.random.randint(NB_LIGNES/2)
print("valeur recherchée pour a1 :", search)

# lecture séquentielle par page
input_data = ......................................(TABLE)
t1 = time.time()
for t in filtrer_table_par_pages(input_data, 1, search):
  print(t)
print("durée :", round(time.time() - t1, 2), "s")


SyntaxError: invalid syntax (ipython-input-3209801944.py, line 7)

## Lecture d'un tuple dans une page

Cette fonction retourne le tuple situé dans la page *num_page* et à la position *position*

In [10]:
def lecture_tuple(table, rowid):
  page_dir = page_dir_name(table)
  page, position = rowid
  with open(os.path.join(page_dir, f"page{page}"), "r") as f:
    lines = f.readlines()
    return lines[position].strip().split(',')


page = np.random.randint(len(os.listdir(page_dir_name(TABLE)))) + 1
position = np.random.randint(1000)
rowid = (page, position)
t1 = time.time()
t = lecture_tuple(TABLE, rowid)
t2 = time.time() - t1
print(f"rowid: {rowid}")
print(f"tuple: {t}")
print("done in", round(t2*1000, 1), "ms")

rowid: (901, 791)
tuple: ['90870', ' 2692190', ' 4406450', ' 1867740', ' 697270', ' 487500', ' 137720', ' ----------------------------------------------------------------------------------------------------']
done in 0.6 ms


# Exercice 1 : Créer un index

## Créer un index unique pour l'attribut $a_0$

On sait que $a_0$ est unique.
Une entrée de l'index associe une *clé* à une *valeur* :
*   La *clé* est l'attribut $a_0$.



In [11]:
def creation_index_unique(table):
  index = {}
  page_dir = page_dir_name(table)

  # nombre de pages
  nb_pages = len(glob.glob(os.path.join(page_dir, "page*")))

  # itérer sur les pages de la table
  # on suppose que les page sont numérotés de 1 à nb_pages
  for page_id in range(1, nb_pages+1) :
    with open(os.path.join(page_dir, f"page{page_id}"), "r") as f:
      for line_number, line in enumerate(f):
        # Convert the key to an integer to match the type of 'a0' generated in the test
        index[int(line.split(',')[0])] = (page_id, line_number)


  # trier les entrées de l'index (nécessaire pour les recherche par intervalle, cf ex 2)
  index = sortedcontainers.SortedDict(index)
  return index

t1 = time.time()
INDEX_UNIQUE_a0 = creation_index_unique(TABLE)
print("durée ", round(time.time() - t1, 3), "s")

durée  5.596 s


In [12]:
# vérifier l'index

# une valeur quelconque pour a0
a0 = MULTIPLE * np.random.randint(NB_LIGNES)

rowid = INDEX_UNIQUE_a0[a0]
print(f"value a0={a0} is in tuple referenced by rowid {rowid}")
print("tuple :", lecture_tuple(TABLE, rowid))

value a0=5185810 is in tuple referenced by rowid (52, 168)
tuple : ['5185810', ' 1908580', ' 4699420', ' 569850', ' 1179460', ' 587820', ' 219460', ' ----------------------------------------------------------------------------------------------------']


## Créer un index non unique pour l'attribut $a_i$

On donne un nom de table et le numéro $i$ de l'attribut $a_i$ de la table.

In [13]:
def creation_index(table, numero_attribut):
  index = {}
  page_dir = page_dir_name(table)

  # nombre de pages
  nb_pages = len(glob.glob(os.path.join(page_dir, "page*")))

  # itérer sur les pages de la table
  # on suppose que les page sont numérotés de 1 à nb_pages
  for page_id in range(1, nb_pages+1) :
    with open(os.path.join(page_dir, f"page{page_id}"), "r") as f:
      for line_number, line in enumerate(f):
        value = int(line.split(',')[numero_attribut])

        if value not in index:
            index[value] = []

        index[value].append((page_id, line_number))

  index = sortedcontainers.SortedDict(index)
  return index


t1 = time.time()
INDEX_a2 = creation_index(TABLE, 2)
print("duree de création de l'index pour l'attribut a2:", round(time.time() - t1, 2), "s")

duree de création de l'index pour l'attribut a2: 7.35 s


In [60]:
# vérifier l'index
s = MULTIPLE * np.random.randint(NB_LIGNES/4)
if s in INDEX_a2:
  print("valeur recherchée :", s)
  for r in INDEX_a2[s]:
    print(r)
else:
  print("valeur non trouvée", s)

valeur recherchée : 479780
(419, 497)
(535, 868)
(749, 144)
(866, 256)
(1511, 433)
(1622, 810)


# Exercice 2 : Accès par index

## Accès par valeur


On veut retrouver les tuples tels qu'un attribut indexé a une valeur donnée.

###Index unique scan
On recherche un unique tuple dont l'attribut indexé a une valeur donnée (car l'attribut est unique)

In [14]:
def acces_par_index_unique(index_unique, table, valeur_recherchee):
  if valeur_recherchee not in index_unique:
    return
  rowid = index_unique[valeur_recherchee]
  return  lecture_tuple(table, rowid)

s = MULTIPLE * np.random.randint(NB_LIGNES)
print("valeur recherchée :", s)

t1 = time.time()
tuple = acces_par_index_unique(INDEX_UNIQUE_a0, TABLE, s)
print("resulat:", tuple)
print("done in", round((time.time() - t1)*1000, 2), "ms")

valeur recherchée : 176420
resulat: ['176420', ' 9407620', ' 1808520', ' 244690', ' 1133890', ' 135840', ' 46220', ' ----------------------------------------------------------------------------------------------------']
done in 2.62 ms


###Index scan
Accès pour rechercher les tuples dont l'attribut indexé a une valeur donnée. On suppose que l'attribut n'est pas unique et que plusieurs tuples sont retrouvés

In [15]:
def acces_par_index(index, table, valeur_recherchee) -> Iterator:
  if valeur_recherchee not in index:
    return
  for rowid in index[valeur_recherchee]:
    tuple =  lecture_tuple(table, rowid)
    yield tuple

s = MULTIPLE * np.random.randint(NB_LIGNES/4)
print("valeur recherchée :", s)

t1 = time.time()
for t in acces_par_index(INDEX_a2, TABLE, s):
  print(t)
print("done in", round((time.time() - t1)*1000, 2), "ms")

valeur recherchée : 1024340
['19789670', ' 7171780', ' 1024340', ' 1961850', ' 226720', ' 100030', ' 13200', ' ----------------------------------------------------------------------------------------------------']
['9579930', ' 3498800', ' 1024340', ' 2429820', ' 467160', ' 245870', ' 61560', ' ----------------------------------------------------------------------------------------------------']
['9462740', ' 3220460', ' 1024340', ' 2340920', ' 826540', ' 15250', ' 69740', ' ----------------------------------------------------------------------------------------------------']
['8798000', ' 4457830', ' 1024340', ' 137380', ' 629300', ' 177020', ' 241870', ' ----------------------------------------------------------------------------------------------------']
['286560', ' 1906500', ' 1024340', ' 1196700', ' 1114250', ' 181910', ' 74650', ' ----------------------------------------------------------------------------------------------------']
done in 5.02 ms


## Accès par intervalle
Index range scan



### Accès par intervalle sur un attribut unique
Définir la fonction *acces_intervalle_par_index_unique* pour rechercher les tuples dont l'attribut indexé est unique et a une valeur comprise dans un intervalle donné.
Indications, votre solution doit prendre en compte les exigences suivantes :

*  Les valeurs recherchées ne sont pas connues à l'avance. On sait seulement qu'elles sont incluses dans un intervalle.

* Ne pas supposer qu'on recherche des entiers consécutifs.

*  Les bornes de l'intervalle ne sont pas parmi les valeurs existantes de l'attribut. Par exemple, on peut rechercher les valeurs de $a_0$ comprises dans l'intervalle  [23 , 45] bien que aucun nuplet n'ait 23 ou 45 comme valeur de $a_0$.


In [17]:
# Exemple pour retrouver la première entrée de l'intervalle [23,45]

indice = INDEX_UNIQUE_a0.bisect_left(23)
print("l'indice de la première clé à retrouver est :", indice)
cle = INDEX_UNIQUE_a0.keys()[indice]
print("la première clé retrouvée est:", cle)
print("la valeur à retrouver est :",  INDEX_UNIQUE_a0[cle])

l'indice de la première clé à retrouver est : 3
la première clé retrouvée est: 30
la valeur à retrouver est : (1783, 941)


In [16]:
def acces_intervalle_par_index_unique(index_unique, table, borne_inf, borne_sup)  -> Iterator:
  indice = index_unique.bisect_left(borne_inf)
  keys = index_unique.keys()

  while (indice < len(keys) and keys[indice] < borne_sup):
    yield acces_par_index_unique(index_unique, table, keys[indice])
    indice += 1

s = MULTIPLE * np.random.randint(nb_valeurs_distinctes/4)
print("valeur recherchée :", s)

t1 = time.time()
for value in acces_intervalle_par_index_unique(INDEX_UNIQUE_a0, TABLE, s + 0.5, s + 10.5):
  print(value)
print("done in", round(time.time() - t1, 2), "s")

valeur recherchée : 20010
['20020', ' 6111240', ' 3000940', ' 2335970', ' 1017120', ' 12350', ' 300970', ' ----------------------------------------------------------------------------------------------------']
done in 0.0 s


### Accès par intervalle sur un attribut NON unique
Définir la fonction *acces_intervalle_par_index* pour rechercher les tuples dont l'attribut indexé n'est **pas** unique et a une valeur comprise dans un intervalle donné.
Votre solution doit prendre en compte les mêmes exigences que celles de la question précédente.

In [18]:
def acces_intervalle_par_index(index, table, borne_inf, borne_sup) -> Iterator:
  indice = index.bisect_left(borne_inf)
  keys = index.keys()

  while (indice < len(keys) and keys[indice] < borne_sup):
      for item in acces_par_index(index, table, keys[indice]):
        yield item
      indice += 1

s = MULTIPLE * np.random.randint(nb_valeurs_distinctes / 4)
print("valeur recherchée :", s)

t1 = time.time()
for value in acces_intervalle_par_index(INDEX_a2, TABLE, s + 3, s + 33):
    print(value)
print("done in", round(time.time() - t1, 2), "s")

valeur recherchée : 65990
['9044570', ' 1207010', ' 66000', ' 1343040', ' 894930', ' 141990', ' 265700', ' ----------------------------------------------------------------------------------------------------']
['6149780', ' 7096940', ' 66000', ' 1216210', ' 31250', ' 571160', ' 149600', ' ----------------------------------------------------------------------------------------------------']
['6405200', ' 1224760', ' 66000', ' 2303270', ' 982260', ' 312790', ' 103820', ' ----------------------------------------------------------------------------------------------------']
['12551330', ' 2270500', ' 66000', ' 2422440', ' 634050', ' 193430', ' 189080', ' ----------------------------------------------------------------------------------------------------']
['1185750', ' 9628400', ' 66010', ' 1128730', ' 1241070', ' 506040', ' 38160', ' ----------------------------------------------------------------------------------------------------']
['3261970', ' 1460590', ' 66010', ' 395220', ' 435190'

# Exercice 3 : Mise à jour de données




## a) Modifier la valeur d'un attribut d'un ou plusieurs tuples

Cela correspond à l'instruction SQL:  UPDATE table SET ... WHERE ...



### Modification d'un seul tuple

On donne une valeur *v* de l'attribut clé $a_0$. Ajouter 20 à l'attribut $a_3$ du nuplet qui a *v* pour valeur de $a_0$ . Cela correspond à l'instruction SQL

UPDATE T
SET a3 = a3 + 20
WHERE a0 = *v*

Après la modification, accéder aux données pour vérifier que le tuple a bien été modifié,  en invoquant la fonction acces_par_index_unique.



In [19]:
def update_unique(index, table, v):
  if v not in index: return

  page_dir = page_dir_name(table)
  rowid = index[v]
  page, position = rowid

  with open(os.path.join(page_dir, f"page{page}"), "r") as f:
    lines = f.readlines()

  my_tuple = lines[position].strip().split(',')
  ancienne_valeur = int(my_tuple[3])
  nouvelle_valeur = ancienne_valeur + 20
  my_tuple[3] = f" {nouvelle_valeur}"

  lines[position] = ','.join(my_tuple) + '\n'

  with open(os.path.join(page_dir, f"page{page}"), "w") as f:
        f.writelines(lines)

print(acces_par_index_unique(INDEX_UNIQUE_a0, TABLE, 14115920))
update_unique(INDEX_UNIQUE_a0, TABLE, 14115920)
print(acces_par_index_unique(INDEX_UNIQUE_a0, TABLE, 14115920))

['14115920', ' 7500410', ' 2947290', ' 1403300', ' 672320', ' 543010', ' 2390', ' ----------------------------------------------------------------------------------------------------']
['14115920', ' 7500410', ' 2947290', ' 1403320', ' 672320', ' 543010', ' 2390', ' ----------------------------------------------------------------------------------------------------']


### Modification de plusieurs tuples

On donne une valeur *v* de l'attribut $a_2$ qui n'est **pas** unique. Ajouter 10 à l'attribut $a_3$ de tous les tuples pour lesquels $a_2 = v$

update T set a3 = a3+10 where a2=v



In [20]:
from operator import index

def update_plusieurs(index, table, v):
  if v not in index: return

  page_dir = page_dir_name(table)
  rowids = index[v]

  for rowid in rowids:
    page, position = rowid

    with open(os.path.join(page_dir, f"page{page}"), "r") as f:
      lines = f.readlines()

    my_tuple = lines[position].strip().split(',')
    ancienne_valeur = int(my_tuple[3])
    nouvelle_valeur = ancienne_valeur + 10
    my_tuple[3] = f" {nouvelle_valeur}"

    lines[position] = ','.join(my_tuple) + '\n'

    with open(os.path.join(page_dir, f"page{page}"), "w") as f:
      f.writelines(lines)

for item in acces_par_index(INDEX_a2, TABLE, 1043880):
  print(item)

update_plusieurs(INDEX_a2, TABLE, 1043880)
print("-----")

for item in acces_par_index(INDEX_a2, TABLE, 1043880):
  print(item)

['9663940', ' 1282880', ' 1043880', ' 1959500', ' 257710', ' 19710', ' 267690', ' ----------------------------------------------------------------------------------------------------']
['14335260', ' 7095680', ' 1043880', ' 366660', ' 259670', ' 272000', ' 303570', ' ----------------------------------------------------------------------------------------------------']
['465330', ' 7620350', ' 1043880', ' 630930', ' 758000', ' 602020', ' 175910', ' ----------------------------------------------------------------------------------------------------']
['15366280', ' 4154330', ' 1043880', ' 2283560', ' 1243990', ' 282230', ' 243170', ' ----------------------------------------------------------------------------------------------------']
['14597920', ' 3233310', ' 1043880', ' 138580', ' 51100', ' 420210', ' 64960', ' ----------------------------------------------------------------------------------------------------']
-----
['9663940', ' 1282880', ' 1043880', ' 1959510', ' 257710', ' 19710'

## b) Modifier l'index en conséquence lorsque l'attribut modifié est indexé
Commencer par créer un index sur l'attribut $a_3$ qu'on appellera *Index_a3*

L'attribut $a_3$ étant maintenant indexé, la mise à jour de la question précédente implique d'actualiser l'index sur $a_3$ pour que les rowid des tuples qui contenaient l'ancienne valeur de $a_3$ soient associés à la nouvelle valeur de $a_3$.

Montrez que votre solution est correcte en affichant une entrée de l'*Index_a3* avant et après sa modification.


In [21]:
Index_a3 = creation_index(TABLE, 3)

# NB1 : pratiquement sûr que ça ne fonctionne pas avec un index unique
# mais on pourrait facilement l'adapter en checkant si on tombe sur une
# liste de tuples ou juste sur un tuple, quand on accède à un élément de l'index
# NB2 : normalement c'est corrigé mais je laisse l'ancien commentaire au cas où
def update_index(index, rowid, old_value, new_value):
  if(isinstance(index[old_value], list)):
    index[old_value].remove(rowid)
    if (len(index[old_value]) == 0): del index[old_value]

    if new_value not in index: index[new_value] = []
    index[new_value].append(rowid)
  elif(isinstance(index[old_value], tuple)):
    del index[old_value]
    index[new_value] = rowid

valeur_a2 = MULTIPLE * np.random.randint(NB_LIGNES/4)
valeur_a3 = lecture_tuple(TABLE, INDEX_a2[valeur_a2][0])[3]

print(Index_a3[int(valeur_a3)])
print("---")
print(Index_a3[int(valeur_a3) + 10])
print("----------------------------------------------")

# Il faudrait idéalement copier-coller le contenu de la fonction update_plusieurs()
# au lieu de l'appeler. Ca permettrait d'itérer sur les rowids de l'index 1 fois au lieu de 2.
# Je ne le fais pas pour garder le code lisible, mais la différence de perf peut être
# très importante sur des tables contenant beaucoup de rowids.
update_plusieurs(INDEX_a2, TABLE, valeur_a2)

for rowid in INDEX_a2[valeur_a2]:
  ancienne_valeur = int(lecture_tuple(TABLE, rowid)[3]) - 10
  nouvelle_valeur = ancienne_valeur + 10
  update_index(Index_a3, rowid, ancienne_valeur, nouvelle_valeur)

print(Index_a3[int(valeur_a3)])
print("---")
print(Index_a3[int(valeur_a3) + 10])

[(480, 589), (666, 987), (941, 587), (1070, 427), (1114, 792), (1178, 277), (1233, 654), (1430, 428), (1613, 651)]
---
[(146, 519), (886, 968), (1273, 395), (1322, 781), (1436, 205), (1458, 150), (1677, 477), (1890, 432)]
----------------------------------------------
[(480, 589), (666, 987), (1070, 427), (1114, 792), (1178, 277), (1233, 654), (1430, 428), (1613, 651)]
---
[(146, 519), (886, 968), (1273, 395), (1322, 781), (1436, 205), (1458, 150), (1677, 477), (1890, 432), (941, 587)]


# Exercice 4 : Persistence des index

Dans cette partie, on veut rendre les index persistents en stockant les entrées triées dans des pages. Cela permet d'utiliser les index plus efficacement en amortissant leur durée de construction.

## a) Stockage d'un index unique

Proposez une solution pour stocker les entrées **triées** d'un index dans plusieurs pages avec une taille de page fixée (10 000 rowids par page).
Etudier le cas d'un index unique et celui d'un index non unique

In [22]:
# NB1 : je sais que les performances sont sûrement très mauvaises
# mais pas le temps de bufferiser pour écrire proprement
# NB2 : normalement c'est corrigé
def store_index(index, table_de_stockage):
  page_dir = page_dir_name(table_de_stockage)
  nb_pages = len(glob.glob(os.path.join(page_dir, "page*")))

  # line_counter = 0
  page_counter = nb_pages + 1
  page_debut = page_counter
  page_buffer = []

  for key in index:
    item = index[key]

    if(isinstance(item, list)):
      for subitem in item:
        page_buffer.append(f"{key}:{subitem}\n")

        if (len(page_buffer) == 10000):
          with open(os.path.join(page_dir, f"page{page_counter}"), "a") as f:
            f.writelines(page_buffer)

          page_buffer = []
          page_counter += 1
    elif(isinstance(item, tuple)):
      page_buffer.append(f"{key}:{item}\n")

      if (len(page_buffer) == 10000):
        with open(os.path.join(page_dir, f"page{page_counter}"), "a") as f:
          f.writelines(page_buffer)

        page_buffer = []
        page_counter += 1

            # if (line_counter == 10000):
    #     page_counter += 1
    #     line_counter = 0
      # with open(os.path.join(page_dir, f"page{page_counter}"), "a") as f:
      #   f.write(str(key) + ":" + str(item))

      # line_counter += 1


        # with open(os.path.join(page_dir, f"page{page_counter}"), "a") as f:
        #   f.write(str(key) + ":" + str(subitem))

        # line_counter += 1

  if (len(page_buffer) > 0):
    with open(os.path.join(page_dir, f"page{page_counter}"), "a") as f:
      f.writelines(page_buffer)

    page_buffer = []
    page_counter += 1

  page_fin = page_counter - 1 # Fix pas propre mais j'ai la flemme de corriger
  return (page_debut, page_fin)

## b) Reconstruction des index

On veut reconstruire des index en exploitant les pages de l'index qui contiennent les entrées triées. Ne **pas** recontruire les mêmes structures (dictionnaire trié associant un attribut à une liste de rowid) que celles de l'exercice 1. Au contraire, pensez à une structure plus compacte qui permet d'accéder rapidement aux pages de l'index.

In [23]:
def extract_index(table_de_stockage, page_debut, page_fin):
  page_dir = page_dir_name(table_de_stockage)
  result = []

  for page in range(page_debut, page_fin + 1):
    with open(os.path.join(page_dir, f"page{page}"), "r") as f:
      first_line = f.readline().split(":")
      min_value_in_page = int(first_line[0])
      result.append(min_value_in_page)

  return result

## c) Adapter en conséquence les opérations de modification de l'index

Illustrer le cas :

update T set a3 = a3+1 where a2=v

où la nouvelle valeur $a_3' = a_3 + 1$ n'est pas déjà présente dans l'index. Il faut donc insérer une nouvelle clé  dans l'index de l'attribut $a_3$.
On suppose qu'il reste de la place dans une page de l'index pour insérer la nouvelle entrée (on peut avoir jusqu'à 12 000 rowids par page d'index).

In [29]:
import bisect

def update_index_page(index_page, rowid, old_v, new_v, table_de_stockage, page_debut, page_fin):
  page_dir = page_dir_name(table_de_stockage)

  # Supprime l'ancienne entrée
  old_page_idx = bisect.bisect_left(index_page, old_v)

  if (old_page_idx == len(index_page)):
    old_page_idx -= 1
  elif (old_page_idx > 0):
    old_page_idx -= 1

  has_been_found = False
  while ((not has_been_found) and  (page_debut + old_page_idx <= page_fin)):
    with open(os.path.join(page_dir, f"page{page_debut + old_page_idx}"), "r") as f:
      lines_old = f.readlines()

    for line in lines_old:
      if (line == (f"{old_v}:{rowid}\n")):
        lines_old.remove(line)

        with open(os.path.join(page_dir, f"page{page_debut + old_page_idx}"), "w") as f:
          f.writelines(lines_old)

        has_been_found = True

    old_page_idx += 1

  # Ajoute la nouvelle entrée
  new_page_idx = bisect.bisect_left(index_page, new_v)

  if (new_page_idx == len(index_page)):
    new_page_idx -= 1
  elif (new_page_idx > 0):
    new_page_idx -= 1

  is_added = False
  while (not is_added):
    with open(os.path.join(page_dir, f"page{page_debut + new_page_idx}"), "r") as f:
      lines_new = f.readlines()

    for idx, line in enumerate(lines_new):
      if (int(line.split(":")[0]) >= new_v):
        lines_new.insert(idx, f"{new_v}:{rowid}\n")
        is_added = True

        with open(os.path.join(page_dir, f"page{page_debut + new_page_idx}"), "w") as f:
          f.writelines(lines_new)

        break

    new_page_idx += 1

def check_index_page(index, valeur):
  page_idx = bisect.bisect_left(index_page, valeur)

  if (page_idx == len(index_page)):
    page_idx -= 1
  elif (page_idx > 0):
    page_idx -= 1

  should_stop = False
  while (not should_stop):
    with open(os.path.join(page_dir, f"page{page_debut + page_idx}"), "r") as f:
      lines = f.readlines()

    for line in lines:
      if (line.startswith(str(valeur))):
        print(line)

      if (int(line.split(":")[0]) > valeur):
        should_stop = True

    page_idx += 1

valeur_a2 = MULTIPLE * np.random.randint(NB_LIGNES/4)
valeur_a3 = int(lecture_tuple(TABLE, INDEX_a2[valeur_a2][0])[3])

page_debut, page_fin = store_index(Index_a3, TABLE)
index_page = extract_index(TABLE, page_debut, page_fin)

# Display le contenu de l'index_page (avant modif)
print("Avant modification")
check_index_page(index_page, valeur_a3)
print("---")
check_index_page(index_page, valeur_a3 + 10)

update_plusieurs(INDEX_a2, TABLE, valeur_a2)

for rowid in INDEX_a2[valeur_a2]:
  ancienne_valeur = int(lecture_tuple(TABLE, rowid)[3]) - 10
  nouvelle_valeur = ancienne_valeur + 10
  update_index_page(index_page, rowid, ancienne_valeur, nouvelle_valeur, TABLE, page_debut, page_fin)

# TODO: Display le contenu de l'index_page (après modif)
print("Après modification")
check_index_page(index_page, valeur_a3)
print("---")
check_index_page(index_page, valeur_a3 + 10)

Avant modification
797160:(140, 772)

797160:(300, 582)

797160:(434, 268)

797160:(451, 500)

797160:(456, 10)

797160:(699, 408)

797160:(719, 16)

797160:(755, 929)

797160:(1118, 374)

797160:(1363, 284)

797160:(1474, 596)

---
797170:(617, 420)

797170:(678, 807)

797170:(840, 999)

797170:(1057, 844)

797170:(1191, 32)

797170:(1198, 561)

797170:(1391, 810)

797170:(1520, 428)

797170:(1647, 627)

797170:(1770, 446)

Après modification
797160:(140, 772)

797160:(300, 582)

797160:(434, 268)

797160:(451, 500)

797160:(456, 10)

797160:(699, 408)

797160:(719, 16)

797160:(1118, 374)

797160:(1363, 284)

797160:(1474, 596)

---
797170:(755, 929)

797170:(617, 420)

797170:(678, 807)

797170:(840, 999)

797170:(1057, 844)

797170:(1191, 32)

797170:(1198, 561)

797170:(1391, 810)

797170:(1520, 428)

797170:(1647, 627)

797170:(1770, 446)



# Exercice 5 : Index bitmap

Cet exercice est moins guidé et plus d'autonomie est attendue.

On considère une table $U(a_0, b_1, b_2)$ contenant $N=10^6$ nuplets.

L'attribut $a_0$ est une clé, il a donc $N$ valeurs distinctes.

Le nombre de valeurs distinctes des attributs $b_1$, $b_2$ est respectivement 10, 100.
Il y a donc en moyenne $10^5$ tuples par valeur de $b_1$ et respectivement $10^4$ tuples par valeur de $b_2$.

On considère la requête SQL, étant donné deux valeurs $v_1$ et $v_2$ :

`SELECT *
FROM U
WHERE b1=v1 AND b2=v2`

On veut evaluer rapidement cette requête en utilisant deux index sur $b_1$ et $b_2$.

L'**objectif** de cet exercice est de proposer une structure d'index qui soit **efficace** pour cette requête.


a) Proposer un index `IndexB1` formé d'un ensemble de 10 *bitmaps* pour l'attribut $b_1$.
* Chaque valeur distincte $v$ de l'attribut indexé est associée à un vecteur de bit (ou bitmap) de dimension égale au nombre de tuples de la table.

* $IndexB1_v[i] = 1 $ ssi le $i^{ème}$ nuplet de la table a la valeur $v$ pour l'attribut $b_1$.

* Constater que bitmaps sont éparses (faible proportion de valeurs non nulles). Cela justifie l'utilisation d'une structure de données compacte pour les représenter (par exemple RoaringBitmap qui réprésente un bimap par l'ensemble des indices $i$ non nuls des nuplets).

b) Mêmes questions pour l'attributs $b_2$

c) En utilisant 2 index bitmap, évaluer la requête ci-dessus pour des valeurs $v_1, v_2$

d) Montrer qu'un index BitMap permet d'évaluer des requêtes de sélection plus rapidement d'un index reposant sur un dictionnaire (cf ex précédents).
Choisir des valeurs $v_1$ et $v_2$ de telle sorte que le résultat de la requête
ne soit pas vide.

Mesurer le gain de performance.





In [ ]:
!pip install -q pyroaring

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 21.7 MB/s eta 0:00:00


In [ ]:
from pyroaring import BitMap

In [ ]:

def créer_bitmap(table, attribut):
  bitmaps_attrib = {}
  page_dir = page_dir_name(table)
  counter = 0;

  # nombre de pages
  nb_pages = len(glob.glob(os.path.join(page_dir, "page*")))

  # itérer sur les pages de la table
  # on suppose que les page sont numérotés de 1 à nb_pages
  for page_id in range(1, nb_pages+1) :
    with open(os.path.join(page_dir, f"page{page_id}"), "r") as f:
      for position, line in enumerate(f):
        assert(position<1000)
        tuple_val = int(line.strip().split(',')[attribut])

        if (tuple_val not in bitmaps_attrib):
          bitmaps_attrib[tuple_val] = BitMap()

        bitmaps_attrib[tuple_val].add(counter)
        counter += 1

  return bitmaps_attrib

index_B1 = créer_bitmap(TABLE,1)
index_B2 = créer_bitmap(TABLE,2)

index_B1_tradi = creation_index(TABLE, 1)
index_B2_tradi = creation_index(TABLE, 2)

# On trouve ici une len différente de 10 (resp. 100) car on est sur la
# table TABLE au lieu de la table U de l'énoncé
print("nombre de bitmaps pour b1", len(index_B1))
print("nombre de bitmaps pour b1", len(index_B2))
print("-----------")

v1 = 8746950
v2 = 754170

debut = time.perf_counter()
requete_1 = index_B1[v1]
requete_2 = index_B2[v2]
requete_finale = requete_1 & requete_2
fin = time.perf_counter()
print(f"Lecture index bitmap ({fin - debut} secondes)")

for tid in requete_finale:
    taille_page = 1000
    # On rajoute 1 car les pages commencent à 1 et pas 0.
    rowid = (int(tid / taille_page) + 1, tid % taille_page)
    print(lecture_tuple(TABLE, rowid))

debut_tradi = time.perf_counter()
requete_1_tradi = index_B1_tradi[v1]
requete_2_tradi = index_B2_tradi[v2]
requete_finale_tradi = list(set(requete_1_tradi) & set(requete_2_tradi))
fin_tradi = time.perf_counter()
print(f"Lecture index traditionel ({fin_tradi - debut_tradi} secondes)")

for rowid in requete_finale_tradi:
  print(lecture_tuple(TABLE, rowid))

nombre de bitmaps pour b1 864486
nombre de bitmaps pour b1 490862
-----------
Lecture index bitmap (0.00016802499999357678 secondes)
['4812300', ' 8746950', ' 754170', ' 322240', ' 379160', ' 207370', ' 242640', ' ----------------------------------------------------------------------------------------------------']
Lecture index traditionel (0.00017191799997817725 secondes)
['4812300', ' 8746950', ' 754170', ' 322240', ' 379160', ' 207370', ' 242640', ' ----------------------------------------------------------------------------------------------------']



# Exercice 6 facultatif: Index couvrant une requête
* Illustrer les cas vus en TD pour lesquels il est possible d'obtenir le résultat d'une requête sans lire les données de la table mais seulement les index.
